# ¿Se pierde calidad por el resumen o por quitar neuronas?

Qwen3.5-0.8B con un índice de **matemáticas** (calibrado con GSM8K train). A 70 %, 50 % y 30 % de las neuronas del MLP compara cinco formas de elegir qué neuronas cargar:

| Criterio | Qué mira |
|---|---|
| `mean` | uso promedio, el criterio actual de `wsi.capture` |
| `energy` | energía (raíz del promedio al cuadrado): pesa más las activaciones grandes |
| `max` | activación máxima: protege a las neuronas raras pero intensas |
| `taylor` | efecto real en la pérdida si se quita la neurona (con gradientes) |
| `*+recon` | el mismo criterio, pero después reajusta las neuronas que quedan para tapar el hueco (**MoI compensado**) |
| `random` | neuronas al azar (el piso) |

Medido con perplejidad (más bajo es mejor) en soluciones de GSM8K test y en Wikipedia.

**Cómo leerlo:** si `taylor` o `+recon` quedan cerca del modelo completo, la pérdida venía del criterio de resumen. Si todos quedan lejos, viene de quitar neuronas.

**Uso:** `Python 3 + GPU L4` → `Ejecutar todo`. Tarda ~15–25 min.

In [ ]:
BUDGETS = "0.7,0.5,0.3"
OUT = "/content/results_index_criteria.json"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U transformers huggingface_hub
!pip install -q flash-linear-attention || echo "fla opcional: sigue sin ella (más lento)"
!python -c "import transformers; print('transformers', transformers.__version__)"

In [ ]:
%%writefile /content/wsi_capture.py
"""Runtime capture -> activation index -> Selective MoI routing.

Two kinds of Mixture of Indexes (MoI) exist in `wsi`:
  * Additive MoI  (`AdditiveMoI` = `DynamicCircuitPager`): the model keeps all its weights; small payloads are ADDED
                   into dormant slots only when the context needs them. Goal: new capabilities, no quality loss.
  * Selective MoI (`SelectiveMoI` = `TaskRouter`, this module): an activation index decides WHICH existing neurons
                   to load; the rest stay out of memory. Goal: fewer resources; costs quality when neurons are dropped.
A Compensated Selective MoI refits the loaded neurons to cover the dropped ones (see
scripts/experiment_index_criteria.py, "+recon").

The rest of `wsi` reads weights only. This module records what the model actually USES when it runs, stores it as an
index per task, and uses those indexes to decide which weights to load.

    1. Capture   `capture_index(model, sequences, task="math")`
                 Streams a corpus through the model with hooks on every linear layer (input and output units) and on
                 every decoder block (residual stream). Per unit it keeps: mean, mean |x|, mean x^2, max |x|, firing
                 rate (|x| > eps) and, for linear inputs, the weight usage  E|x_i| * ||W[:, i]||.
                 `trace_all(model, ids)` instead records every leaf module output densely for a few tokens
                 (hidden states, attention weights and the KV / recurrent cache included when the model returns them).
    2. Index     `ActivationIndex`: save / load (safetensors + json), `select()` the MLP neurons to keep under a
                 budget, `overlap()` and `specific_units()` to compare tasks, `head_usage()` for attention heads.
    3. Route     `TaskRouter(indexes)`: profiles a prompt, picks the closest task index, and `loaded()` physically
                 slices the MLP matrices to that index's neurons (a real smaller model) and restores them after.

Works with nn.Linear and GPT-2 style Conv1D (weight [in, out]); MLPs are found structurally (a module whose "down"
linear reads exactly what its sibling "up"/"gate" linears write), so Llama/Qwen (gate/up/down), LFM3 (w1/w3/w2),
GPT-2 (c_fc/c_proj) and OPT (fc1/fc2) need no per-model code.
"""
from __future__ import annotations

import json
from contextlib import contextmanager
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Iterable, Iterator

import torch
import torch.nn as nn

STATS = ("n", "sum", "sum_abs", "sum_sq", "max_abs", "fire")
_EPS_FIRE = 1e-6


# ----------------------------------------------------------------------------------------------- module helpers
def _is_linear(m: nn.Module) -> bool:
    return isinstance(m, nn.Linear) or (type(m).__name__ == "Conv1D" and getattr(m, "weight", None) is not None
                                        and m.weight.dim() == 2)


def _io_dims(m: nn.Module) -> tuple[int, int]:
    """(in_features, out_features) for Linear ([out, in] weight) and Conv1D ([in, out] weight)."""
    w = m.weight
    return (w.shape[1], w.shape[0]) if isinstance(m, nn.Linear) else (w.shape[0], w.shape[1])


def _input_col_norms(m: nn.Module) -> torch.Tensor:
    """||W[:, i]||: how strongly input unit i is written into the output."""
    w = m.weight.float()
    return w.norm(dim=0) if isinstance(m, nn.Linear) else w.norm(dim=1)


def find_blocks(model: nn.Module) -> tuple[str, nn.ModuleList]:
    """The decoder stack: the largest ModuleList in the model."""
    best = max(((n, m) for n, m in model.named_modules() if isinstance(m, nn.ModuleList)),
               key=lambda nm: len(nm[1]), default=(None, None))
    if best[1] is None:
        raise ValueError("no ModuleList of decoder blocks found")
    return best


@dataclass
class MLPGroup:
    name: str            # parent module
    ups: list[str]       # linears that write the hidden units (gate, up / w1, w3 / c_fc / fc1)
    down: str            # linear that reads them (down_proj / w2 / c_proj / fc2)
    units: int


def find_mlps(model: nn.Module) -> list[MLPGroup]:
    groups = []
    for pname, parent in model.named_modules():
        lin = {n: c for n, c in parent.named_children() if _is_linear(c)}
        if len(lin) < 2:
            continue
        for dn, d in lin.items():
            d_in, d_out = _io_dims(d)
            if d_in <= d_out:  # the down projection contracts the (wider) hidden units back to the residual width
                continue
            ups = [n for n, c in lin.items() if n != dn and _io_dims(c) == (d_out, d_in)]
            if ups and len(ups) == len(lin) - 1:
                pre = f"{pname}." if pname else ""
                groups.append(MLPGroup(pname, [pre + u for u in ups], pre + dn, d_in))
                break
    return groups


def _tensor_out(o: Any) -> torch.Tensor | None:
    if isinstance(o, torch.Tensor):
        return o
    if isinstance(o, (tuple, list)) and o and isinstance(o[0], torch.Tensor):
        return o[0]
    return getattr(o, "last_hidden_state", None) if not isinstance(o, dict) else None


# ----------------------------------------------------------------------------------------------- statistics
class _Acc:
    def __init__(self, dim: int, device: torch.device):
        z = lambda: torch.zeros(dim, device=device, dtype=torch.float64)  # noqa: E731
        self.n = torch.zeros((), device=device, dtype=torch.float64)
        self.sum, self.sum_abs, self.sum_sq, self.max_abs, self.fire = z(), z(), z(), z(), z()

    def add(self, x: torch.Tensor) -> None:
        x = x.detach().reshape(-1, x.shape[-1]).float()
        a = x.abs()
        self.n += x.shape[0]
        self.sum += x.sum(0).double()
        self.sum_abs += a.sum(0).double()
        self.sum_sq += (x * x).sum(0).double()
        self.max_abs = torch.maximum(self.max_abs, a.max(0).values.double())
        self.fire += (a > _EPS_FIRE).sum(0).double()

    def tensors(self) -> dict[str, torch.Tensor]:
        return {k: getattr(self, k).cpu() for k in STATS}


# ----------------------------------------------------------------------------------------------- the index
@dataclass
class ActivationIndex:
    """What a model used on one task: per-unit activation statistics for every captured site."""

    task: str
    tokens: int
    stats: dict[str, dict[str, torch.Tensor]]      # site -> {n, sum, sum_abs, sum_sq, max_abs, fire}
    usage: dict[str, torch.Tensor]                  # linear name -> E|x_i| * ||W[:, i]|| per input unit
    mlps: list[MLPGroup] = field(default_factory=list)
    meta: dict[str, Any] = field(default_factory=dict)

    # -- per-site views
    def mean_abs(self, site: str) -> torch.Tensor:
        s = self.stats[site]
        return s["sum_abs"] / s["n"].clamp_min(1)

    def firing_rate(self, site: str) -> torch.Tensor:
        s = self.stats[site]
        return s["fire"] / s["n"].clamp_min(1)

    def rms(self, site: str) -> torch.Tensor:
        s = self.stats[site]
        return (s["sum_sq"] / s["n"].clamp_min(1)).sqrt()

    def neuron_scores(self) -> dict[str, torch.Tensor]:
        """MLP neuron importance: weight usage of the down projection's input units."""
        return {g.down: self.usage[g.down] for g in self.mlps}

    def head_usage(self, linear: str, n_heads: int) -> torch.Tensor:
        """Share of usage per attention head, from the input units of an output projection (o_proj / out_proj)."""
        u = self.usage[linear]
        h = u.reshape(n_heads, -1).sum(1)
        return h / h.sum().clamp_min(1e-12)

    # -- choosing what to load
    def select(self, budget: float, mode: str = "uniform") -> dict[str, torch.Tensor]:
        """MLP neurons to keep: {down linear name: sorted unit indices}.

        uniform: the same fraction `budget` of neurons in every layer.
        global : ranks all neurons together by their share of their layer's usage, so layers whose usage is spread
                 out keep more neurons (total = budget * all neurons; each layer keeps at least one).
        """
        scores = self.neuron_scores()
        if mode == "uniform":
            return {d: s.topk(max(1, round(budget * len(s)))).indices.sort().values for d, s in scores.items()}
        if mode != "global":
            raise ValueError("mode must be 'uniform' or 'global'")
        names = list(scores)
        rel = [scores[d] / scores[d].sum().clamp_min(1e-12) for d in names]
        flat = torch.cat(rel)
        k = max(len(names), round(budget * flat.numel()))
        keep = torch.zeros_like(flat, dtype=torch.bool)
        keep[flat.topk(k).indices] = True
        out, off = {}, 0
        for d, r in zip(names, rel):
            m = keep[off : off + len(r)]
            off += len(r)
            out[d] = m.nonzero().flatten() if m.any() else r.argmax().reshape(1)
        return out

    def overlap(self, other: "ActivationIndex", budget: float) -> float:
        """Fraction of selected neurons shared with another task's index at the same budget."""
        a, b = self.select(budget), other.select(budget)
        inter = sum(len(set(a[d].tolist()) & set(b[d].tolist())) for d in a)
        return inter / max(sum(len(v) for v in a.values()), 1)

    def specific_units(self, other: "ActivationIndex", top: int = 20) -> list[dict[str, Any]]:
        """Neurons this task uses much more than `other` (log ratio of weight usage)."""
        rows = []
        for d, s in self.neuron_scores().items():
            r = (s + 1e-9).log() - (other.usage[d] + 1e-9).log()
            for i in r.topk(min(top, len(r))).indices.tolist():
                rows.append({"linear": d, "unit": i, "log_ratio": float(r[i]),
                             "usage_self": float(s[i]), "usage_other": float(other.usage[d][i])})
        return sorted(rows, key=lambda x: -x["log_ratio"])[:top]

    def profile(self) -> torch.Tensor:
        """Task fingerprint: per-layer normalized mean |h| of every MLP hidden unit, concatenated."""
        v = [self.mean_abs(g.down + ":in") for g in self.mlps]
        return torch.cat([x / x.norm().clamp_min(1e-12) for x in v])

    def size_bytes(self) -> int:
        return sum(t.numel() * t.element_size() for s in self.stats.values() for t in s.values()) + \
            sum(t.numel() * t.element_size() for t in self.usage.values())

    # -- persistence
    def save(self, path: str | Path) -> Path:
        from safetensors.torch import save_file

        p = Path(path)
        p.mkdir(parents=True, exist_ok=True)
        flat = {f"stats/{site}/{k}": v.contiguous() for site, s in self.stats.items() for k, v in s.items()}
        flat.update({f"usage/{n}": v.contiguous() for n, v in self.usage.items()})
        save_file(flat, str(p / "activations.safetensors"))
        (p / "index.json").write_text(json.dumps({
            "task": self.task, "tokens": self.tokens, "meta": self.meta,
            "mlps": [g.__dict__ for g in self.mlps]}, indent=2), encoding="utf-8")
        return p

    @classmethod
    def load(cls, path: str | Path) -> "ActivationIndex":
        from safetensors.torch import load_file

        p = Path(path)
        info = json.loads((p / "index.json").read_text(encoding="utf-8"))
        stats: dict[str, dict[str, torch.Tensor]] = {}
        usage: dict[str, torch.Tensor] = {}
        for k, v in load_file(str(p / "activations.safetensors")).items():
            kind, rest = k.split("/", 1)
            if kind == "usage":
                usage[rest] = v
            else:
                site, stat = rest.rsplit("/", 1)
                stats.setdefault(site, {})[stat] = v
        return cls(info["task"], info["tokens"], stats, usage, [MLPGroup(**g) for g in info["mlps"]], info["meta"])


# ----------------------------------------------------------------------------------------------- capture
@torch.no_grad()
def capture_index(
    model: nn.Module,
    sequences: Iterable[list[int] | torch.Tensor],
    *,
    task: str = "task",
    max_tokens: int | None = None,
    linear_outputs: bool = True,
    max_out_dim: int = 65_536,
) -> ActivationIndex:
    """Stream `sequences` (token id lists, one forward each) through `model` and build its ActivationIndex.

    Sites captured: "<linear>:in" and "<linear>:out" for every linear layer (outputs wider than `max_out_dim`, i.e.
    the LM head, are skipped), and "<block>:resid" for the residual stream after every decoder block.
    """
    dev = next(model.parameters()).device
    accs: dict[str, _Acc] = {}
    handles = []

    def site(name: str, dim: int) -> _Acc:
        if name not in accs:
            accs[name] = _Acc(dim, dev)
        return accs[name]

    linears = {n: m for n, m in model.named_modules() if _is_linear(m)}
    for n, m in linears.items():
        d_in, d_out = _io_dims(m)

        def hook(mod, inp, out, n=n, d_in=d_in, d_out=d_out):
            site(f"{n}:in", d_in).add(inp[0])
            if linear_outputs and d_out <= max_out_dim:
                site(f"{n}:out", d_out).add(out)

        handles.append(m.register_forward_hook(hook))
    bname, blocks = find_blocks(model)
    for i, b in enumerate(blocks):
        def bhook(mod, inp, out, i=i):
            t = _tensor_out(out)
            if t is not None:
                site(f"{bname}.{i}:resid", t.shape[-1]).add(t)

        handles.append(b.register_forward_hook(bhook))

    seen = 0
    try:
        for s in sequences:
            ids = s if isinstance(s, torch.Tensor) else torch.tensor(s)
            ids = ids.reshape(1, -1).to(dev)
            if max_tokens is not None and seen >= max_tokens:
                break
            model(input_ids=ids)
            seen += ids.shape[1]
    finally:
        for h in handles:
            h.remove()

    stats = {k: a.tensors() for k, a in accs.items()}
    usage = {}
    for n, m in linears.items():
        if f"{n}:in" in stats:
            s = stats[f"{n}:in"]
            usage[n] = (s["sum_abs"] / s["n"].clamp_min(1)).float() * _input_col_norms(m).cpu()
    cfg = getattr(model, "config", None)
    meta = {"model": getattr(cfg, "_name_or_path", None) or type(model).__name__,
            "model_type": getattr(cfg, "model_type", None), "blocks": bname}
    return ActivationIndex(task, seen, stats, usage, find_mlps(model), meta)


@torch.no_grad()
def trace_all(model: nn.Module, input_ids: torch.Tensor | list[int], *, max_tokens: int = 32) -> dict[str, torch.Tensor]:
    """Dense record of EVERYTHING for a short input: every leaf module output, every linear input, and whatever the
    model returns (hidden states, attention weights when the attention implementation exposes them, KV / recurrent
    cache tensors). Size grows with tokens x all widths, so keep inputs short."""
    dev = next(model.parameters()).device
    ids = (input_ids if isinstance(input_ids, torch.Tensor) else torch.tensor(input_ids)).reshape(1, -1)[:, :max_tokens]
    rec: dict[str, torch.Tensor] = {}
    handles = []
    for n, m in model.named_modules():
        if not n:
            continue
        leaf = not any(True for _ in m.children())
        if not (leaf or _is_linear(m)):
            continue

        def hook(mod, inp, out, n=n, lin=_is_linear(m)):
            t = _tensor_out(out)
            if t is not None:
                rec[f"{n}:out"] = t.detach().float().cpu()
            if lin and inp and isinstance(inp[0], torch.Tensor):
                rec[f"{n}:in"] = inp[0].detach().float().cpu()

        handles.append(m.register_forward_hook(hook))
    try:
        try:
            out = model(input_ids=ids.to(dev), output_hidden_states=True, output_attentions=True, use_cache=True)
        except (TypeError, ValueError, NotImplementedError):
            out = model(input_ids=ids.to(dev), output_hidden_states=True, use_cache=True)
    finally:
        for h in handles:
            h.remove()
    for i, h in enumerate(getattr(out, "hidden_states", None) or ()):
        rec[f"hidden_states.{i}"] = h.float().cpu()
    for i, a in enumerate(getattr(out, "attentions", None) or ()):
        if isinstance(a, torch.Tensor):
            rec[f"attentions.{i}"] = a.float().cpu()
    cache = getattr(out, "past_key_values", None)
    for name in ("key_cache", "value_cache", "conv_states", "recurrent_states", "ssm_states"):
        for i, t in enumerate(getattr(cache, name, None) or ()):
            if isinstance(t, torch.Tensor) and t.numel():
                rec[f"cache.{name}.{i}"] = t.float().cpu()
    for i, lyr in enumerate(getattr(cache, "layers", None) or ()):
        for attr in ("keys", "values", "conv_states", "recurrent_states"):
            t = getattr(lyr, attr, None)
            if isinstance(t, torch.Tensor) and t.numel():
                rec[f"cache.layers.{i}.{attr}"] = t.float().cpu()
    return rec


# ----------------------------------------------------------------------------------------------- loading an index
@contextmanager
def apply_selection(model: nn.Module, selection: dict[str, torch.Tensor]) -> Iterator[dict[str, Any]]:
    """Physically slice every selected MLP to its kept neurons (up/gate output rows, down input columns), run the
    body, then restore the full weights. Yields the parameter counts before and after."""
    mods = dict(model.named_modules())
    groups = {g.down: g for g in find_mlps(model)}
    saved = []
    before = sum(p.numel() for p in model.parameters())

    def put(m: nn.Module, w: torch.Tensor, b: torch.Tensor | None) -> None:
        saved.append((m, m.weight, m.bias, getattr(m, "in_features", None), getattr(m, "out_features", None),
                      getattr(m, "nf", None)))
        m.weight = nn.Parameter(w.contiguous(), requires_grad=False)
        if b is not None:
            m.bias = nn.Parameter(b.contiguous(), requires_grad=False)
        i, o = _io_dims(m)
        if isinstance(m, nn.Linear):
            m.in_features, m.out_features = i, o
        else:
            m.nf = o

    try:
        for down, ix in selection.items():
            g = groups[down]
            ix = ix.to(mods[down].weight.device)
            for u in g.ups:
                m = mods[u]
                w = m.weight[ix] if isinstance(m, nn.Linear) else m.weight[:, ix]
                put(m, w, None if m.bias is None else m.bias[ix])
            d = mods[down]
            put(d, d.weight[:, ix] if isinstance(d, nn.Linear) else d.weight[ix], None)
        yield {"params_full": before, "params_loaded": sum(p.numel() for p in model.parameters())}
    finally:
        for m, w, b, i, o, nf in reversed(saved):
            m.weight = w
            if b is not None or m.bias is not None:
                m.bias = b
            if isinstance(m, nn.Linear):
                m.in_features, m.out_features = i, o
            elif nf is not None:
                m.nf = nf


class TaskRouter:
    """Selective MoI at the task level: pick which activation index (and so which neurons) to load per prompt.

    Alias: `SelectiveMoI`. For adding capabilities without removing anything, use Additive MoI
    (`wsi.AdditiveMoI` / `DynamicCircuitPager`)."""

    def __init__(self, indexes: dict[str, ActivationIndex] | list[ActivationIndex]):
        self.indexes = indexes if isinstance(indexes, dict) else {i.task: i for i in indexes}
        self._profiles = {k: v.profile() for k, v in self.indexes.items()}

    @torch.no_grad()
    def route(self, model: nn.Module, input_ids: list[int] | torch.Tensor) -> dict[str, Any]:
        """Profile the prompt with one prefill (MLP hidden mean |h| only) and return the closest task by cosine."""
        probe = capture_index(model, [input_ids], task="prompt", linear_outputs=False)
        p = probe.profile()
        scores = {k: float(torch.nn.functional.cosine_similarity(p, v, dim=0)) for k, v in self._profiles.items()}
        best = max(scores, key=scores.get)
        return {"task": best, "scores": scores}

    @contextmanager
    def loaded(self, model: nn.Module, task: str, budget: float, mode: str = "uniform") -> Iterator[dict[str, Any]]:
        with apply_selection(model, self.indexes[task].select(budget, mode)) as info:
            yield {"task": task, "budget": budget, "mode": mode, **info}


SelectiveMoI = TaskRouter

In [ ]:
%%writefile /content/experiment_index_criteria.py
"""Is the quality lost by a task index due to the SUMMARY (selection criterion) or to removing neurons at all?

Same setup as experiment_capture_index_route.py (Qwen3.5-0.8B, math index from GSM8K train, physical MLP slicing),
with five ways to pick the kept neurons at each budget, all computed in one calibration pass:
  * mean     : E|h_i| * ||w_i||           (the current wsi.capture weight usage; w_i = down-proj column i)
  * energy   : sqrt(E h_i^2) * ||w_i||    (weights rare-but-large activations more than the mean does)
  * max      : max|h_i| * ||w_i||         (protects neurons that fire rarely but strongly)
  * taylor   : E_seq |sum_t h_i(t) g_i(t)|, g = dLoss/dh  (first-order loss change if neuron i is removed)
  * +recon   : a criterion's selection, then the kept down-proj columns are refit by ridge least squares so the
               layer reproduces the full MLP output on the calibration data:  W_S = W G_{:,S} (G_SS + lam I)^-1,
               G = E[h h^T]. Changes only weights that stay loaded (compensation, as in SparseGPT-style pruning).
Scored by teacher-forced perplexity on held-out GSM8K test solutions (in-domain) and WikiText-2 test (side effect).

    python scripts/experiment_index_criteria.py
"""
from __future__ import annotations

import argparse
import json
import math
import re
import sys
import time
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT))
try:
    from wsi.capture import apply_selection, find_mlps  # noqa: E402
except ImportError:  # Colab: wsi/capture.py copied next to this script as wsi_capture.py
    from wsi_capture import apply_selection, find_mlps  # noqa: E402

HUB_FILES = {
    "gsm8k/test.parquet": ("openai/gsm8k", "main/test-00000-of-00001.parquet"),
    "gsm8k/train.parquet": ("openai/gsm8k", "main/train-00000-of-00001.parquet"),
    "wikitext/test.parquet": ("Salesforce/wikitext", "wikitext-2-raw-v1/test-00000-of-00001.parquet"),
}


def need(data_dir: Path, name: str) -> Path:
    path = data_dir / name
    if path.exists():
        return path
    from huggingface_hub import hf_hub_download

    repo, fn = HUB_FILES[name]
    got = Path(hf_hub_download(repo, fn, repo_type="dataset", local_dir=data_dir / "_hub"))
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(got.read_bytes())
    return path


def clean_solution(ans: str) -> str:
    body, final = ans.split("####")
    return re.sub(r"<<[^>]*>>", "", body).strip() + f"\nThe answer is {final.strip()}."


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--model-dir", default=str(ROOT / "models" / "qwen35-0.8b-full"))
    ap.add_argument("--data-dir", default=str(ROOT / "work"))
    ap.add_argument("--calib-tokens", type=int, default=60_000)
    ap.add_argument("--n-eval", type=int, default=60)
    ap.add_argument("--budgets", default="0.7,0.5,0.3")
    ap.add_argument("--taylor-seqs", type=int, default=150, help="sequences used for the gradient (Taylor) pass")
    ap.add_argument("--ridge", type=float, default=1e-2, help="lambda as a fraction of mean diag(G)")
    ap.add_argument("--out", default=str(ROOT / "benchmarks" / "index_criteria" / "results.json"))
    args = ap.parse_args()

    torch.manual_seed(0)
    dev = torch.device("cuda")
    data = Path(args.data_dir)
    tok = AutoTokenizer.from_pretrained(args.model_dir)
    model = AutoModelForCausalLM.from_pretrained(args.model_dir, dtype=torch.bfloat16).to(dev).eval()
    for p in model.parameters():
        p.requires_grad_(False)
    mods = dict(model.named_modules())
    groups = find_mlps(model)
    downs = [g.down for g in groups]
    U = groups[0].units
    print(f"{len(groups)} MLPs x {U} neurons")

    def chat(q: str, a: str) -> list[int]:
        txt = tok.apply_chat_template([{"role": "user", "content": q}, {"role": "assistant", "content": a}],
                                      tokenize=False, enable_thinking=False)
        return tok(txt, add_special_tokens=False).input_ids

    gtr = pd.read_parquet(need(data, "gsm8k/train.parquet"))
    gte = pd.read_parquet(need(data, "gsm8k/test.parquet")).iloc[: args.n_eval]
    calib, n = [], 0
    for q, a in zip(gtr["question"], gtr["answer"]):
        calib.append(chat(q, clean_solution(a)))
        n += len(calib[-1])
        if n >= args.calib_tokens:
            break
    wt = tok("\n".join(pd.read_parquet(need(data, "wikitext/test.parquet"))["text"]), add_special_tokens=False).input_ids
    held = {"math": [chat(q, clean_solution(a)) for q, a in zip(gte["question"], gte["answer"])],
            "wiki": [wt[i : i + 256] for i in range(0, 256 * args.n_eval, 256)]}

    # ---------------------------------------------------------------- one calibration pass (forward + backward)
    st = {d: {"abs": torch.zeros(U, device=dev, dtype=torch.float64),
              "sq": torch.zeros(U, device=dev, dtype=torch.float64),
              "max": torch.zeros(U, device=dev, dtype=torch.float64),
              "taylor": torch.zeros(U, device=dev, dtype=torch.float64),
              "G": torch.zeros(U, U, device=dev, dtype=torch.float32)} for d in downs}
    leaves: dict[str, torch.Tensor] = {}

    def pre_hook(name):
        def hook(mod, inp):
            h = inp[0].detach().requires_grad_(True)  # leaf: gives dLoss/dh without parameter grads
            leaves[name] = h
            x = h.detach().reshape(-1, U).float()
            s = st[name]
            s["abs"] += x.abs().sum(0).double()
            s["sq"] += (x * x).sum(0).double()
            s["max"] = torch.maximum(s["max"], x.abs().max(0).values.double())
            s["G"] += x.T @ x
            return (h,) + tuple(inp[1:])
        return hook

    state = {"grad": False}

    def stat_or_grad_hook(name):
        inner = pre_hook(name)

        def hook(mod, inp):
            if state["grad"]:  # gradient pass: only make h a leaf, no statistics
                h = inp[0].detach().requires_grad_(True)
                leaves[name] = h
                return (h,) + tuple(inp[1:])
            inner(mod, inp)
            return None
        return hook

    hs = [mods[d].register_forward_pre_hook(stat_or_grad_hook(d)) for d in downs]
    t0 = time.perf_counter()
    ntok = 0
    with torch.no_grad():  # pass 1: activation statistics and Gram matrices over all calibration tokens
        for s in calib:
            model(input_ids=torch.tensor([s], device=dev), use_cache=False)
            ntok += len(s)
    leaves.clear()
    print(f"pass 1: {ntok:,} math tokens ({len(calib)} sequences) in {time.perf_counter() - t0:.0f}s, "
          f"GPU {torch.cuda.memory_allocated() / 2**30:.1f} GB", flush=True)
    # pass 2: Taylor scores need dLoss/dh; a subset of sequences is enough and keeps memory bounded
    state["grad"] = True
    grad_seqs = calib[: args.taylor_seqs]
    t1 = time.perf_counter()
    for j, s in enumerate(grad_seqs):
        ids = torch.tensor([s], device=dev)
        with torch.enable_grad():
            logits = model(input_ids=ids, use_cache=False).logits[0, :-1].float()
            F.cross_entropy(logits, ids[0, 1:], reduction="sum").backward()
        for d in downs:
            h = leaves[d]
            st[d]["taylor"] += (h.detach().float() * h.grad.float()).reshape(-1, U).sum(0).abs().double()
        leaves.clear()
        del logits
        if (j + 1) % 50 == 0:
            print(f"  pass 2: {j + 1}/{len(grad_seqs)} seqs, GPU {torch.cuda.memory_allocated() / 2**30:.1f} GB "
                  f"(peak {torch.cuda.max_memory_allocated() / 2**30:.1f})", flush=True)
    state["grad"] = False
    for h in hs:
        h.remove()
    torch.cuda.empty_cache()
    print(f"pass 2: Taylor on {len(grad_seqs)} sequences in {time.perf_counter() - t1:.0f}s", flush=True)

    wnorm = {d: mods[d].weight.float().norm(dim=0).double() for d in downs}
    score = {
        "mean": {d: st[d]["abs"] / ntok * wnorm[d] for d in downs},
        "energy": {d: (st[d]["sq"] / ntok).sqrt() * wnorm[d] for d in downs},
        "max": {d: st[d]["max"] * wnorm[d] for d in downs},
        "taylor": {d: st[d]["taylor"] / len(grad_seqs) for d in downs},
    }

    def select(crit: str, b: float) -> dict[str, torch.Tensor]:
        k = round(b * U)
        return {d: score[crit][d].topk(k).indices.sort().values for d in downs}

    def recon_weight(d: str, ix: torch.Tensor) -> torch.Tensor:
        G = st[d]["G"].double() / ntok
        W = mods[d].weight.double()  # [hidden, U] (full, before slicing)
        Gss = G[ix][:, ix]
        lam = args.ridge * Gss.diagonal().mean()
        rhs = W @ G[:, ix]  # [hidden, |S|]
        Ws = torch.linalg.solve(Gss + lam * torch.eye(len(ix), device=dev, dtype=torch.float64), rhs.T).T
        return Ws.to(mods[d].weight.dtype)

    @torch.no_grad()
    def ppl(seqs) -> float:
        nll, nt = 0.0, 0
        for s in seqs:
            ids = torch.tensor([s], device=dev)
            lp = F.log_softmax(model(input_ids=ids).logits[0, :-1].float(), dim=-1)
            nll -= float(lp.gather(1, ids[0, 1:, None]).sum())
            nt += len(s) - 1
        return math.exp(nll / nt)

    def evaluate(sel: dict | None, recon: bool = False) -> dict:
        if sel is None:
            return {t: round(ppl(s), 3) for t, s in held.items()}
        new_w = {d: recon_weight(d, ix.to(dev)) for d, ix in sel.items()} if recon else {}
        with apply_selection(model, sel):
            for d, w in new_w.items():
                mods[d].weight.data.copy_(w)
            return {t: round(ppl(s), 3) for t, s in held.items()}

    res = {"model": args.model_dir, "calib_tokens": ntok, "full": evaluate(None), "budgets": {}}
    print(f"[full] math={res['full']['math']:.3f} wiki={res['full']['wiki']:.3f}", flush=True)
    for b in [float(x) for x in args.budgets.split(",")]:
        row = {}
        for crit in ("mean", "energy", "max", "taylor"):
            sel = select(crit, b)
            row[crit] = evaluate(sel)
            row[crit + "+recon"] = evaluate(sel, recon=True)
        g = torch.Generator().manual_seed(0)
        row["random"] = evaluate({d: torch.randperm(U, generator=g)[: round(b * U)].sort().values for d in downs})
        ref = select("mean", b)
        row["overlap_with_mean"] = {c: round(sum(len(set(select(c, b)[d].tolist()) & set(ref[d].tolist()))
                                                 for d in downs) / (len(downs) * round(b * U)), 4)
                                    for c in ("energy", "max", "taylor")}
        res["budgets"][str(b)] = row
        print(f"\n=== {b:.0%} of neurons ===   (math full {res['full']['math']:.2f}, wiki full {res['full']['wiki']:.2f})")
        for c, v in row.items():
            if c != "overlap_with_mean":
                print(f"  {c:14s} math={v['math']:9.3f}  wiki={v['wiki']:9.3f}")
        print("  overlap with 'mean':", row["overlap_with_mean"], flush=True)
    out = Path(args.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(res, indent=2), encoding="utf-8")
    print("Saved", out)


if __name__ == "__main__":
    main()

In [ ]:
!cd /content && python -u experiment_index_criteria.py --model-dir Qwen/Qwen3.5-0.8B --data-dir /content/data     --budgets {BUDGETS} --out {OUT}

In [ ]:
import json
r = json.load(open(OUT))
f = r["full"]
print(f"Modelo completo: ppl math {f['math']:.2f} | ppl wiki {f['wiki']:.2f}\n")
for b, row in r["budgets"].items():
    print(f"=== {float(b):.0%} de las neuronas ===")
    print(f"{'criterio':16s} {'ppl math':>10s} {'ppl wiki':>10s}")
    for c, v in sorted(((c, v) for c, v in row.items() if c != "overlap_with_mean"), key=lambda cv: cv[1]["math"]):
        print(f"{c:16s} {v['math']:10.2f} {v['wiki']:10.2f}")
    print("solapamiento con 'mean':", row["overlap_with_mean"], "\n")
from google.colab import files
files.download(OUT)